# Ocean Plastic Waste Classification — Model Evaluation & Verification Pipeline
### Sprint 1: Foundation | Module: S1-EVAL
**Nhóm Dự Án:** Phân loại và giám sát rác thải nhựa đại dương (TrashCan 1.0 — Instance)  
**Tác giả:** Nhóm Đánh giá & Tối ưu hóa (Evaluation & Optimization)  
**Phối hợp:** Phan Hải Yến (S1-ARCH / Lead), Hoàng Bách (S1-CNN / Training), Hùng (S1-DATA / Preprocessing), Ngọc Nguyên (S1-DEMO / Inference)  
**Tài liệu kế hoạch đánh giá:** [docs/evaluation_plan.md](../docs/evaluation_plan.md)

---
### Mục Tiêu Notebook
1. **Kiểm tra và tích hợp toàn diện module `src/evaluate.py`** với cả môi trường Google Colab và Local.
2. **Xác minh công thức toán học** của các chỉ số đánh giá (Accuracy, Precision, Recall, F1-score, Macro F1, Weighted F1) bằng dữ liệu giả định có kết quả kỳ vọng đã biết.
3. **Trực quan hóa ma trận nhầm lẫn (Confusion Matrix)** và phát hiện các cặp lớp rác thải dễ bị nhầm lẫn nhất.
4. **Trực quan hóa đường cong học tập (Learning Curves)** từ dữ liệu lịch sử huấn luyện mẫu, tự động đánh dấu Best Epoch.
5. **Thực thi quy trình phân tích lỗi (Error Analysis)**: Khảo sát các mẫu dự đoán sai trên ảnh mô phỏng môi trường quang học dưới nước (độ đục, ánh sáng xanh lam/lục, mất màu đỏ).
6. **Xây dựng khung so sánh thực nghiệm (Experiment Comparison)** giữa CNN Baseline và các phương án cải tiến (Data Augmentation, Transfer Learning).
7. **Lưu vết kết quả (Reproducibility)** ra file JSON và Markdown sẵn sàng phục vụ báo cáo.

> **Lưu ý quan trọng:** Toàn bộ dữ liệu trong notebook này là **dữ liệu mẫu (Synthetic Smoke Test Data)** dùng để xác minh tính đúng đắn và độ tin cậy của code pipeline. Dữ liệu này **không phải và không được trình bày là kết quả thực nghiệm chính thức** trên bộ dữ liệu TrashCan 1.0 (kết quả chính thức sẽ được tổng hợp khi Hoàng Bách hoàn thành huấn luyện CNN Baseline).

In [ ]:
# ==============================================================================
# CẤU HÌNH MÔI TRƯỜNG & TỰ ĐỘNG PHÁT HIỆN GOOGLE COLAB
# ==============================================================================
import os
import sys

# Bảo đảm in ký tự tiếng Việt an toàn trên mọi hệ điều hành
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Kiểm tra xem có đang chạy trên Google Colab hay không
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[Colab] Đang chạy trên môi trường Google Colab.")
    # Thêm thư mục làm việc vào sys.path nếu cần
    if not os.path.exists("src"):
        print("[Colab] Cấu hình đường dẫn root cho Google Colab...")
        # !git clone https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git
        # %cd ocean-plastic-ml-starter
else:
    print("[Local] Đang chạy trên môi trường máy cục bộ (Local Environment).")

# Đảm bảo thư mục gốc dự án nằm trong sys.path để import src
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Xác định thư mục results/ chuẩn ở project root
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"[Setup] Thư mục gốc dự án: {PROJECT_ROOT}")
print(f"[Setup] Thư mục lưu artifacts (results): {RESULTS_DIR}")
print(f"[Setup] Python version: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# IMPORT THƯ VIỆN & MODULE SRC/EVALUATE
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import các hàm đánh giá cốt lõi từ src/evaluate
from src.evaluate import (
    compute_classification_metrics,
    generate_classification_report,
    compute_confusion_matrix,
    plot_confusion_matrix,
    plot_learning_curves,
    identify_misclassified_samples,
    find_top_confused_pairs,
    plot_misclassified_samples,
    format_comparison_table,
    save_evaluation_results,
    smoke_test,
)

print("[Success] Toàn bộ các hàm từ src/evaluate đã được import thành công!")

## 1. Thiết Lập Class Mapping (TrashCan 1.0 — Instance)

Trong bài toán phân loại rác thải biển, danh mục nhãn được mô phỏng theo cấu trúc của bộ dữ liệu **TrashCan 1.0 — Instance**. Để kiểm tra toàn diện quy trình đánh giá trong Sprint 1, chúng ta sử dụng tập 6 lớp rác thải và đối tượng biển tiêu biểu:
1. `trash_bag` (Túi nilon rác trôi nổi / quấn đá)
2. `trash_bottle` (Chai nhựa nước ngọt, hóa chất)
3. `trash_can` (Lon kim loại, hộp nhôm phế thải)
4. `trash_net` (Lưới đánh cá cũ, dây thừng phế thải)
5. `trash_wrapper` (Vỏ bao bì bánh kẹo, túi bọc thực phẩm)
6. `biological_or_rov` (Sinh vật biển, san hô, hoặc thiết bị ROV)

> **Hợp đồng giao diện (Interface Contract) với Hoàng Bách (S1-CNN):**  
> Khi mô hình hoàn thành dự đoán trên tập kiểm thử (Test Set):  
> - Đầu ra nhãn thực tế: `y_true` (mảng 1D các số nguyên từ $0$ đến $C-1$).  
> - Đầu ra nhãn dự đoán: `y_pred` (mảng 1D sau hàm `argmax`).  
> - Đầu ra phân bố xác suất: `y_probs` (mảng 2D kích thước $(N, C)$ với $\sum_{c} P_c = 1.0$).

In [ ]:
# ==============================================================================
# KHỞI TẠO CLASS MAPPING & DỮ LIỆU KIỂM THỬ MẪU (SMOKE TEST DATA)
# ==============================================================================
TARGET_NAMES = [
    "trash_bag",
    "trash_bottle",
    "trash_can",
    "trash_net",
    "trash_wrapper",
    "biological_or_rov",
]
NUM_CLASSES = len(TARGET_NAMES)
CLASS_TO_IDX = {name: i for i, name in enumerate(TARGET_NAMES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(TARGET_NAMES)}

print("Danh mục nhãn đánh giá (Class Mapping):")
for idx, name in IDX_TO_CLASS.items():
    print(f"  [{idx}] -> {name}")

# Khởi tạo tập dữ liệu mẫu gồm 120 mẫu có phân bố mất cân bằng (Class Imbalance)
# tương tự như dữ liệu thực tế dưới đáy biển
np.random.seed(42)

# Phân bố mẫu: trash_bag (35), trash_bottle (30), trash_can (15), trash_net (15), trash_wrapper (15), bio (10)
y_true = np.array(
    [0] * 35 + [1] * 30 + [2] * 15 + [3] * 15 + [4] * 15 + [5] * 10
)
total_samples = len(y_true)

# Mô phỏng nhãn dự đoán y_pred với tỷ lệ đúng khoảng 85%
# Các lỗi phân loại được thiết kế có chủ đích phản ánh nhầm lẫn thực tế
y_pred = y_true.copy()

# Giả lập một số lỗi:
# - Một số túi nilon (trash_bag: 0) bị đoán nhầm thành lưới (trash_net: 3) hoặc vỏ bao bì (trash_wrapper: 4)
y_pred[0:4] = 3
y_pred[4:7] = 4

# - Một số chai nhựa (trash_bottle: 1) bị đoán nhầm thành lon (trash_can: 2) hoặc túi (trash_bag: 0)
y_pred[35:38] = 2
y_pred[38:40] = 0

# - Một số lon (trash_can: 2) bị đoán nhầm thành chai (trash_bottle: 1)
y_pred[65:68] = 1

# - Một số sinh vật (bio: 5) bị đoán nhầm thành rác (trash_net: 3 hoặc trash_bag: 0)
y_pred[110:112] = 3
y_pred[112:113] = 0

# Sinh ma trận xác suất softmax mô phỏng (y_probs)
y_probs = np.zeros((total_samples, NUM_CLASSES), dtype=np.float32)
for i in range(total_samples):
    pred_c = y_pred[i]
    high_prob = np.random.uniform(0.65, 0.95)
    remaining = (1.0 - high_prob) / (NUM_CLASSES - 1)
    y_probs[i, :] = remaining
    y_probs[i, pred_c] = high_prob

print(f"\n[Data] Tổng số mẫu kiểm thử mẫu: {total_samples}")
print(f"[Data] Số mẫu phân loại đúng: {np.sum(y_true == y_pred)} / {total_samples}")
print(f"[Data] Số mẫu phân loại sai: {np.sum(y_true != y_pred)} / {total_samples}")

## 2. Xác Minh Công Thức Toán Học & Tính Toán Chỉ Số (Metrics Verification)

### Bản chất của các chỉ số:
1. **Accuracy:** Tỷ lệ số mẫu đúng trên tổng số mẫu: $\frac{\sum TP_i}{N}$.
2. **Precision:** Khả năng dự đoán chính xác khi khẳng định một lớp: $\frac{TP_i}{TP_i + FP_i}$ (Tránh báo động giả / tránh gắp nhầm sinh vật biển).
3. **Recall:** Khả năng phát hiện trọn vẹn rác thải có trong thực tế: $\frac{TP_i}{TP_i + FN_i}$ (Tránh bỏ sót rác thải trôi nổi).
4. **Macro F1-score (Primary Metric):** $\frac{1}{C}\sum_{i=1}^C F_{1, i}$ — Đánh giá công bằng mọi lớp, phạt nặng nếu mô hình bỏ quên lớp hiếm.
5. **Weighted F1-score:** $\sum_{i=1}^C \frac{N_i}{N} F_{1, i}$ — Phản ánh hiệu năng theo tỷ lệ mẫu thực tế.

Chúng ta sẽ chạy `compute_classification_metrics` và đối chiếu với công thức chuẩn.

In [ ]:
# ==============================================================================
# TÍNH TOÁN BỘ CHỈ SỐ ĐÁNH GIÁ VÀ XUẤT CLASSIFICATION REPORT
# ==============================================================================
metrics_result = compute_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
    target_names=TARGET_NAMES,
)

print("=" * 65)
print("KẾT QUẢ ĐÁNH GIÁ TỔNG QUAN (OVERALL EVALUATION METRICS)")
print("=" * 65)
print(f"Tổng số mẫu kiểm thử (Total Samples):      {metrics_result['total_samples']}")
print(f"Overall Accuracy:                          {metrics_result['accuracy']:.4f} ({metrics_result['accuracy']:.2%})")
print(f"Macro Precision:                           {metrics_result['macro_precision']:.4f}")
print(f"Macro Recall:                              {metrics_result['macro_recall']:.4f}")
print(f">>> Macro F1-score (PRIMARY METRIC):       {metrics_result['macro_f1']:.4f} <<<")
print(f"Weighted Precision:                        {metrics_result['weighted_precision']:.4f}")
print(f"Weighted Recall:                           {metrics_result['weighted_recall']:.4f}")
print(f"Weighted F1-score:                         {metrics_result['weighted_f1']:.4f}")

print("\n" + "=" * 65)
print("BÁO CÁO PHÂN LOẠI CHI TIẾT THEO TỪNG LỚP (CLASSIFICATION REPORT)")
print("=" * 65)
report_text = generate_classification_report(
    y_true=y_true,
    y_pred=y_pred,
    target_names=TARGET_NAMES,
    digits=4,
)
print(report_text)

## 3. Trực Quan Hóa Ma Trận Nhầm Lẫn (Confusion Matrix)

Confusion Matrix biểu diễn trực quan mối tương quan giữa **Nhãn Thực Tế (True Class)** ở trục tung và **Nhãn Dự Đoán (Predicted Class)** ở trục hoành.
- **Đường chéo chính:** Các mẫu phân loại đúng ($TP$).
- **Ngoài đường chéo chính:** Các trường hợp nhầm lẫn.
- Ta hiển thị 2 phiên bản:
  1. **Raw Counts:** Số lượng mẫu cụ thể.
  2. **Normalized by True Class:** Tỷ lệ phần trăm trên mỗi hàng (tương ứng Recall của từng lớp rác thải).

In [ ]:
# ==============================================================================
# TÍNH TOÁN VÀ VẼ CONFUSION MATRIX (RAW COUNTS & NORMALIZED)
# ==============================================================================
# 1. Tính toán ma trận nhầm lẫn dạng thô (raw counts)
cm_raw = compute_confusion_matrix(y_true, y_pred)

# 2. Vẽ ma trận chuẩn hóa và lưu file artifact
cm_norm_path = os.path.join(RESULTS_DIR, "confusion_matrix_normalized.png")
fig_cm, ax_cm = plot_confusion_matrix(
    cm=cm_raw,
    target_names=TARGET_NAMES,
    normalize=True,
    title="Underwater Trash Classification Confusion Matrix",
    cmap="Blues",
    figsize=(9, 7),
    save_path=cm_norm_path,
    show=True,
)

# 3. Vẽ ma trận dạng đếm số lượng mẫu (Raw Counts)
cm_raw_path = os.path.join(RESULTS_DIR, "confusion_matrix_raw.png")
fig_cm_raw, ax_cm_raw = plot_confusion_matrix(
    cm=cm_raw,
    target_names=TARGET_NAMES,
    normalize=False,
    title="Underwater Trash Classification Confusion Matrix",
    cmap="Blues",
    figsize=(9, 7),
    save_path=cm_raw_path,
    show=True,
)

## 4. Trực Quan Hóa Đường Cong Học Tập (Learning Curves: Loss & Accuracy)

Đường cong học tập biểu diễn sự thay đổi của hàm mất mát (Loss) và độ chính xác (Accuracy) trên cả tập Training và Validation qua từng Epoch:
- **Hiện tượng Overfitting:** Thể hiện khi Training Loss tiếp tục giảm nhưng Validation Loss bắt đầu đi ngang hoặc tăng trở lại.
- **Best Epoch Selection:** Hàm `plot_learning_curves` tự động xác định và đánh dấu điểm dừng tối ưu (epoch có `val_loss` thấp nhất và `val_accuracy` cao nhất). Đây chính là checkpoint sẽ được lưu vào `best_model.keras`.

In [ ]:
# ==============================================================================
# MÔ PHỎNG LỊCH SỬ HUẤN LUYỆN (TRAINING HISTORY) & VẼ LEARNING CURVES
# ==============================================================================
# Dữ liệu mẫu mô phỏng 25 epoch huấn luyện CNN Baseline:
# - Ban đầu (Ep 1-10): Học nhanh, loss giảm mạnh từ 1.8 xuống 0.5.
# - Giai đoạn tối ưu (Ep 15-18): Val loss đạt cực tiểu (khoảng 0.38 tại epoch 17).
# - Giai đoạn sau (Ep 19-25): Bắt đầu xuất hiện overfitting nhẹ, val loss dao động tăng lên 0.45.
epochs_range = list(range(1, 26))

mock_history = {
    "loss": [
        1.85, 1.50, 1.25, 1.05, 0.90, 0.78, 0.68, 0.60, 0.53, 0.47,
        0.42, 0.38, 0.34, 0.31, 0.28, 0.25, 0.23, 0.20, 0.18, 0.16,
        0.14, 0.13, 0.11, 0.10, 0.09,
    ],
    "val_loss": [
        1.88, 1.55, 1.30, 1.10, 0.95, 0.82, 0.72, 0.65, 0.58, 0.52,
        0.48, 0.45, 0.42, 0.40, 0.39, 0.385, 0.380, 0.390, 0.405, 0.420,
        0.435, 0.445, 0.460, 0.470, 0.485,
    ],
    "accuracy": [
        0.30, 0.42, 0.52, 0.61, 0.68, 0.73, 0.78, 0.81, 0.84, 0.86,
        0.88, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.965, 0.97,
        0.975, 0.98, 0.985, 0.99, 0.992,
    ],
    "val_accuracy": [
        0.28, 0.40, 0.50, 0.58, 0.65, 0.70, 0.75, 0.78, 0.81, 0.83,
        0.85, 0.86, 0.87, 0.875, 0.88, 0.885, 0.890, 0.885, 0.880, 0.875,
        0.870, 0.865, 0.860, 0.855, 0.850,
    ],
}

# Vẽ đường cong học tập và lưu file artifact
lc_path = os.path.join(RESULTS_DIR, "learning_curves.png")
fig_lc, (ax_loss, ax_acc) = plot_learning_curves(
    history=mock_history,
    save_path=lc_path,
    title_prefix="CNN Baseline (Smoke Test)",
    figsize=(14, 5),
    show=True,
)

## 5. Quy Trình Phân Tích Lỗi (Error Analysis) & Đặc Thù Môi Trường Biển

Quy trình phân tích lỗi bao gồm:
1. **Lọc các cặp nhầm lẫn phổ biến nhất (Top Confused Pairs):** Xác định mô hình thường phân vân giữa những loại rác nào.
2. **Khảo sát độ tin cậy (Confidence Analysis):** Phân biệt giữa lỗi mô hình tự tin cao (High-confidence errors: $P > 0.80$) và lỗi mô hình phân vân (Low-confidence errors: $P < 0.50$).
3. **Trực quan hóa mẫu ảnh bị phân loại sai:** Kiểm tra trực quan xem hình thái rác, độ đục nước, hoặc ánh sáng có phải là nguyên nhân gây lỗi hay không.

In [ ]:
# ==============================================================================
# TRÍCH XUẤT CÁC CẶP NHẦM LẪN HÀNG ĐẦU (TOP CONFUSED PAIRS)
# ==============================================================================
top_confused = find_top_confused_pairs(cm_raw, target_names=TARGET_NAMES, top_k=5)

print("=" * 70)
print("DANH SÁCH TOP CÁC CẶP LỚP BỊ NHẦM LẪN NHIỀU NHẤT (TOP CONFUSED PAIRS)")
print("=" * 70)
print(f"{'Hạng':<5} | {'Nhãn Thực Tế':<20} -> {'Nhãn Dự Đoán':<20} | {'Số Mẫu':<8} | {'Tỷ Lệ Nhầm'}")
print("-" * 70)
for rank, item in enumerate(top_confused, 1):
    print(
        f"{rank:<5} | {item['true_name']:<20} -> {item['pred_name']:<20} | "
        f"{item['count']:<8} | {item['percentage_of_true']:.1%}"
    )

In [ ]:
# ==============================================================================
# KHẢO SÁT CHI TIẾT TỪNG MẪU DỰ ĐOÁN SAI & PHÂN HẠNG MỨC ĐỘ LỖI
# ==============================================================================
mock_metadata = [{"image_id": f"frame_{i:04d}.jpg", "video_id": f"ROV_clip_{i//15:02d}"} for i in range(total_samples)]

misclassified_records = identify_misclassified_samples(
    y_true=y_true,
    y_pred=y_pred,
    probabilities=y_probs,
    target_names=TARGET_NAMES,
    sample_metadata=mock_metadata,
)

print(f"Tổng số mẫu dự đoán sai: {len(misclassified_records)} / {total_samples}")

# Thống kê mức độ nghiêm trọng của lỗi
severity_counts = pd.Series([rec["severity"] for rec in misclassified_records]).value_counts()
print("\nPhân bố mức độ lỗi:")
for sev, cnt in severity_counts.items():
    print(f"  - {sev}: {cnt} mẫu")

# Hiển thị 5 ví dụ lỗi đầu tiên
print("\n5 mẫu lỗi tiêu biểu:")
sample_err_df = pd.DataFrame([
    {
        "ID": r["metadata"]["image_id"],
        "Video": r["metadata"]["video_id"],
        "Nhãn Thực": r["true_class_name"],
        "Nhãn Đoán": r["pred_class_name"],
        "Độ Tin Cậy": f"{r['confidence']:.2%}",
        "Mức Độ": r["severity"],
    }
    for r in misclassified_records[:5]
])
print(sample_err_df.to_string(index=False))

### Trực Quan Hóa Mẫu Ảnh Bị Phân Loại Sai (Mô Phỏng Quang Học Dưới Nước)

Dưới đây ta tạo các mảng ảnh mẫu mô phỏng điều kiện quang học thực tế dưới đáy biển:
- **Tông màu nước biển xanh lam / xanh lục (Greenish/Cyan Tint)** do nước hấp thụ quang phổ đỏ.
- **Độ đục và tán xạ ánh sáng (Turbidity / Blur / Backscatter)**.
- Hiển thị lưới $3 \times 3$ các mẫu bị phân loại sai để nhóm nghiên cứu cùng phân tích nguyên nhân.

In [ ]:
# ==============================================================================
# TẠO ẢNH GIẢ LẬP ĐẶC THÙ NƯỚC BIỂN & HIỂN THỊ LƯỚI PHÂN TÍCH LỖI
# ==============================================================================
# Tạo ảnh RGB 64x64 mô phỏng quang học đáy biển
mock_images = []
for i in range(total_samples):
    # Nền nước biển: kênh Red thấp (0.05-0.2), kênh Green và Blue cao (0.4-0.8)
    base_water = np.zeros((64, 64, 3), dtype=np.float32)
    base_water[:, :, 0] = np.random.uniform(0.05, 0.20)  # Red bị hấp thụ mạnh
    base_water[:, :, 1] = np.random.uniform(0.35, 0.65)  # Green cao
    base_water[:, :, 2] = np.random.uniform(0.50, 0.85)  # Blue cao

    # Thêm nhiễu đục / hạt lơ lửng (Turbidity noise)
    turbidity = np.random.normal(0, 0.05, (64, 64, 3)).astype(np.float32)
    img = np.clip(base_water + turbidity, 0.0, 1.0)

    # Thêm hình khối tượng trưng cho vật thể rác thải ở giữa ảnh
    c = y_true[i]
    if c == 0:  # trash_bag: vùng sáng mờ loang lổ
        img[20:45, 20:45, :] += 0.25
    elif c == 1:  # trash_bottle: hình chữ nhật dọc
        img[15:50, 25:38, :] += 0.35
    elif c == 2:  # trash_can: hình chữ nhật nhỏ đậm
        img[22:42, 24:40, :] += 0.30
    elif c == 3:  # trash_net: các đường sọc
        img[15:50, 15:50:5, :] += 0.28
    elif c == 4:  # trash_wrapper: hình vuông phẳng
        img[22:42, 22:42, :] += 0.22
    else:  # bio/rov
        img[18:48, 18:48, 0] += 0.30  # Chứa chút sắc tố đỏ hoặc sinh vật

    mock_images.append(np.clip(img, 0.0, 1.0))

mock_images = np.array(mock_images)

# Trực quan hóa 9 mẫu dự đoán sai đầu tiên
err_grid_path = os.path.join(RESULTS_DIR, "misclassified_samples.png")
fig_err, axes_err = plot_misclassified_samples(
    images=mock_images,
    misclassified_info=misclassified_records,
    target_names=TARGET_NAMES,
    max_samples=9,
    save_path=err_grid_path,
    figsize=(11, 11),
    show=True,
)

## 6. Thiết Kế So Sánh Thực Nghiệm (Experiment Comparison)

Trong nghiên cứu khoa học, ta cần so sánh mô hình cơ sở (**CNN Baseline**) với các kỹ thuật cải tiến trong điều kiện kiểm soát nghiêm ngặt:
- **Cùng tập Test Video-Level** (Không thay đổi dữ liệu test).
- **Cùng Random Seed = 42**.
- **Cùng kích thước ảnh 224x224**.
- **Chỉ số quyết định:** **Macro F1-score**.

Dưới đây là mẫu bảng so sánh các kịch bản nghiên cứu được định dạng bằng hàm `format_comparison_table`.

In [ ]:
# ==============================================================================
# KHỞI TẠO BẢNG SO SÁNH CÁC THỰC NGHIỆM ĐỀ XUẤT (COMPARISON TABLE)
# ==============================================================================
# Danh sách kịch bản thực nghiệm mẫu
comparison_experiments = [
    {
        "exp_id": "EXP-001",
        "model_name": "CNN Baseline",
        "augmentation": "None (Resize only)",
        "val_loss": 0.3800,
        "test_loss": 0.3950,
        "test_accuracy": 0.8583,
        "test_macro_precision": 0.8410,
        "test_macro_recall": 0.8350,
        "test_macro_f1": 0.8380,
        "parameters": "2.4M",
        "notes": "Mô hình cơ sở của Hoàng Bách (Benchmark)",
    },
    {
        "exp_id": "EXP-002",
        "model_name": "CNN Baseline + Underwater Aug",
        "augmentation": "Color Jitter + Blur + Random Rotation",
        "val_loss": 0.3120,
        "test_loss": 0.3250,
        "test_accuracy": 0.8917,
        "test_macro_precision": 0.8850,
        "test_macro_recall": 0.8790,
        "test_macro_f1": 0.8820,
        "parameters": "2.4M",
        "notes": "Mô phỏng quang phổ và độ đục nước biển",
    },
    {
        "exp_id": "EXP-003",
        "model_name": "Transfer Learning (MobileNetV2)",
        "augmentation": "Underwater Aug + Pretrained ImageNet",
        "val_loss": 0.2450,
        "test_loss": 0.2580,
        "test_accuracy": 0.9250,
        "test_macro_precision": 0.9180,
        "test_macro_recall": 0.9210,
        "test_macro_f1": 0.9195,
        "parameters": "3.5M",
        "notes": "Kịch bản đề xuất cho Sprint 2",
    },
]

comparison_md = format_comparison_table(comparison_experiments, output_format="markdown")
print("BẢNG SO SÁNH CÁC KỊCH BẢN THỰC NGHIỆM (MARKDOWN TABLE):")
print(comparison_md)

## 7. Xuất Kết Quả Đánh Giá Ra File (Export & Reproducibility)

Theo nguyên tắc tái lập của ML, mọi kết quả đánh giá cần được lưu vết dưới dạng:
- File cấu trúc máy đọc: `results/evaluation_smoke_test_results.json`
- Báo cáo văn bản người đọc: `results/evaluation_smoke_test_summary.md`

In [ ]:
# ==============================================================================
# LƯU KẾT QUẢ ĐÁNH GIÁ VÀO THƯ MỤC RESULTS/
# ==============================================================================
json_export_path = os.path.join(RESULTS_DIR, "evaluation_smoke_test_results.json")
md_export_path = os.path.join(RESULTS_DIR, "evaluation_smoke_test_summary.md")

save_evaluation_results(metrics_result, json_export_path, format="json")
save_evaluation_results(metrics_result, md_export_path, format="markdown")

print(f"\n[Artifacts] Kiểm tra danh sách file đã tạo trong {RESULTS_DIR}:")
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  - results/{fname:<35} ({size_kb:.2f} KB)")

## 8. Kết Luận & Biên Bản Bàn Giao (Handover Checklist)

### Bảng Kiểm Tra Tiêu Chí Hoàn Thành (Definition of Done — Acceptance Criteria)

| Tiêu Chí Chấp Nhận | Trạng Thái | Minh Chứng / Ghi Chú |
|---|---|---|
| **1. Kế hoạch đánh giá chi tiết** | ĐÃ ĐẠT | Tài liệu hoàn chỉnh tại [docs/evaluation_plan.md](../docs/evaluation_plan.md). |
| **2. Xác định các chỉ số đánh giá** | ĐÃ ĐẠT | Accuracy, Precision, Recall, F1-score và Macro F1 làm Primary Metric. |
| **3. Module đánh giá `src/evaluate.py`** | ĐÃ ĐẠT | 9 hàm cốt lõi, xử lý ngoại lệ, docstring đầy đủ. |
| **4. Trực quan hóa Confusion Matrix** | ĐÃ ĐẠT | Xuất file `results/confusion_matrix_normalized.png` và `raw.png`. |
| **5. Trực quan hóa Learning Curves** | ĐÃ ĐẠT | Biểu đồ Loss và Accuracy, tự động tìm Best Epoch tại `results/learning_curves.png`. |
| **6. Khung phân tích lỗi (Error Analysis)** | ĐÃ ĐẠT | Trích xuất Top Confused Pairs, khảo sát mẫu lỗi trên ảnh mô phỏng nước biển tại `results/misclassified_samples.png`. |
| **7. Mẫu so sánh thực nghiệm** | ĐÃ ĐẠT | Hàm `format_comparison_table` tạo bảng khoa học so sánh Baseline vs Cải tiến. |
| **8. Tương thích Google Colab & Local** | ĐÃ ĐẠT | Tự động phát hiện môi trường, cấu hình `sys.path` an toàn. |
| **9. Không sử dụng số liệu giả định làm kết quả thật** | ĐÃ ĐẠT | Toàn bộ dữ liệu trong notebook được ghi chú rõ ràng là Synthetic Smoke Test Data. |
| **10. Bàn giao giao diện cho Hoàng Bách và Yến** | SẴN SÀNG | Sẵn sàng tạo Pull Request để nhóm review. |

---
**Cam kết:** Evaluation Pipeline đã sẵn sàng 100% để tiếp nhận mô hình CNN Baseline được huấn luyện chính thức từ Hoàng Bách!